# OCR Bilans Fiscaux Algériens — V16 — JSON + contrôles + liasse Excel

> V16 = V13 + sous-totaux avec bons libellés (modèle Excel) + contrôles de sous-totaux + écarts.

## Apports V16
- Schéma ACTIF enrichi des sous-totaux du modèle : `immobilisations_corporelles`, `immobilisations_financieres`, `creances_et_emplois_assimiles`, `disponibilites_et_assimiles`.
- Formules de contrôle des sous-totaux (somme des composantes) → écart extrait/recalculé tracé.
- Résolution des lignes Excel par libellé normalisé (robuste aux décalages de lignes), sous-totaux = cellules formule → jamais écrites, seulement comparées.
- Écarts reportés : JSON (`controles`), feuille `0. Données extraites`, classeur (bordure rouge + commentaire).

## Règle d'or
Ne jamais inventer de valeur : absent/illisible → null.

In [ ]:
%pip install -q -U 'transformers>=4.57.0' accelerate pymupdf pillow psutil
print('✅ OK')

In [ ]:
import time, json, re, gc, copy, unicodedata, shutil, subprocess, sys
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import Counter
import fitz, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
import openpyxl
from openpyxl.comments import Comment
from openpyxl.styles import Font, PatternFill, Border, Side, Protection
print('✅ Imports OK')

In [ ]:
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_NEW_TOKENS = 4096
IMAGE_MAX_SIZE = 2024
MIN_PIXELS = 4*32*32
MAX_PIXELS = 2000*32*32
PDF_ZOOM = 3.0
BLANK_THRESHOLD = 0.95
CLASSIF_BATCH_SIZE = 16
GPU_BATCH_SIZE = 4
CLASSIF_RES = 1200
TOLERANCE_DA = 1.0
INPUT_DIR = Path('/mnt/Risk/bilans_in')
OUTPUT_DIR = Path('/mnt/Risk/bilans_out')
JSON_DIR = OUTPUT_DIR / 'json_bilans_v16'
CONTROLE_DIR = OUTPUT_DIR / 'json_controles_v16'
XLSX_DIR = OUTPUT_DIR / 'liasses_xlsx_v16'
LOG_PATH = OUTPUT_DIR / 'pipeline_bilans_v16.log'
for d in (OUTPUT_DIR, JSON_DIR, CONTROLE_DIR, XLSX_DIR): d.mkdir(parents=True, exist_ok=True)
MODELE_XLSX = Path('/mnt/Risk/Model/Liasse_fiscale_G2_BNP_Paribas_El_Djazair.xlsx')
RECALC_TOOL = Path('/mnt/Risk/Model/recalc_liasse.py')
RECALC_DISPONIBLE = RECALC_TOOL.exists() and bool(shutil.which('soffice'))
pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print('Device:', DEVICE, '| PDF:', len(pdfs))

In [ ]:
def log(msg):
    ligne = datetime.now().strftime('%Y-%m-%d %H:%M:%S') + ' — ' + str(msg)
    print(ligne, flush=True)
    with open(LOG_PATH, 'a', encoding='utf-8') as f: f.write(ligne + chr(10))
print('✅ Log OK')

In [ ]:
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = 'left'
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config
model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, dtype=torch.bfloat16, device_map='auto', trust_remote_code=True, low_cpu_mem_usage=True, quantization_config=FP8Config(dequantize=True))
model.eval()
print('✅ Modèle chargé en ' + str(round(time.time()-t0,1)) + 's')

In [ ]:
def chunks(lst, n):
    for i in range(0, len(lst), n): yield lst[i:i+n]
def resize(img, max_side=IMAGE_MAX_SIZE):
    w,h = img.size
    if max(w,h) <= max_side: return img
    r = max_side/max(w,h); return img.resize((int(w*r), int(h*r)), Image.LANCZOS)
def strip_accents(s): return ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))
def norm_key(s): return re.sub('[^a-z0-9]+','_', strip_accents(s).lower()).strip('_')
def estimate_skew(img):
    small = img.convert('L').copy(); small.thumbnail((500,500))
    def score(a):
        r = np.array(small.rotate(a, expand=True, fillcolor=255)) < 128
        return float(((r.sum(axis=1))**2).sum())
    best = max(range(-12,13,2), key=score)
    best = max([best-1,best-0.5,best,best+0.5,best+1], key=score)
    return best if abs(best) >= 1 else 0.0
def deskew(img):
    a = estimate_skew(img)
    if a: img = img.rotate(a, expand=True, fillcolor=(255,255,255), resample=Image.BICUBIC)
    return img
def is_blank(image, threshold=BLANK_THRESHOLD):
    arr = np.array(image.convert('L')); return (arr > 240).sum()/arr.size >= threshold
def pdf_to_pages(path, zoom=PDF_ZOOM):
    doc = fitz.open(path); matrix = fitz.Matrix(zoom, zoom); pages = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
        pages.append({'index': i, 'image': resize(deskew(img))})
    doc.close(); return pages
def parse_json(text):
    if not text: return {}
    try: return json.loads(text.strip())
    except Exception: pass
    m = re.search(r'\{.*\}', text, re.S)
    try: return json.loads(m.group()) if m else {}
    except Exception: return {}
def apply_template(messages):
    try: return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError: return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
def _decode(o, n): return processor.decode(o[n:], skip_special_tokens=True, clean_up_tokenization_spaces=False)
def ask_single(prompt, image):
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    inputs = processor(text=[apply_template(msgs)], images=[image], return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    return {'text': _decode(out[0], inputs['input_ids'].shape[1]), 'tokens_in': int(inputs['input_ids'].shape[1]), 'tokens_out': int(out[0].shape[0]-inputs['input_ids'].shape[1])}
def ask_batch(prompt, images):
    if not images: return []
    if len(images) == 1: return [ask_single(prompt, images[0])]
    msgs = [[{'role':'user','content':[{'type':'image','image':im},{'type':'text','text':prompt}]}] for im in images]
    inputs = processor(text=[apply_template(m) for m in msgs], images=images, return_tensors='pt', padding=True).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    el = time.time()-t0; n = inputs['input_ids'].shape[1]; attn = inputs.get('attention_mask')
    return [{'text': _decode(out[i], n), 'tokens_in': int(attn[i].sum().item()) if attn is not None else n, 'tokens_out': int(out[i].shape[0]-n)} for i in range(len(images))]
print('✅ Utilitaires OK')

In [ ]:
# SCHÉMAS — V16 : sous-totaux ACTIF ajoutés avec libellés du modèle
SCHEMAS = {
 'ACTIF': {'cols': ['montant_brut','amortissements_provisions_pertes','net_n','net_n1'], 'postes': {
   'ecarts_acquisition_goodwill':'Ecart d acquisition goodwill','immobilisations_incorporelles':'Immobilisations incorporelles',
   'immobilisations_corporelles':'Immobilisations corporelles','terrains':'Terrains','batiments':'Batiments',
   'autres_immobilisations_corporelles':'Autres immobilisations corporelles','immobilisations_en_concession':'Immobilisations en concession',
   'immobilisations_en_cours':'Immobilisations en cours','immobilisations_financieres':'Immobilisations financieres',
   'titres_mis_en_equivalence':'Titres mis en equivalence','autres_participations_creances':'Autres participations et creances rattachees',
   'autres_titres_immobilises':'Autres titres immobilises','prets_actifs_financiers_non_courants':'Prets et autres actifs financiers non courants',
   'impots_differes_actif':'Impots differes actif','total_actif_non_courant':'TOTAL ACTIF NON COURANT',
   'stocks_encours':'Stocks et encours','creances_et_emplois_assimiles':'Creances et emplois assimiles',
   'clients':'Clients','autres_debiteurs':'Autres debiteurs','impots_assimiles_actif':'Impots et assimiles',
   'autres_creances_assimiles':'Autres creances et emplois assimiles','disponibilites_et_assimiles':'Disponibilites et assimiles',
   'placements_financiers_courants':'Placements et autres actifs financiers courants','tresorerie_actif':'Tresorerie',
   'total_actif_courant':'TOTAL ACTIF COURANT','total_general_actif':'TOTAL GENERAL ACTIF'}},
 'PASSIF': {'cols': ['n','n1'], 'postes': {
   'capital_emis':'Capital emis','capital_non_appele':'Capital non appele','primes_reserves':'Primes et reserves',
   'ecart_reevaluation':'Ecarts de reevaluation','ecart_equivalence':'Ecart d equivalence','resultat_net_passif':'Resultat net',
   'report_a_nouveau':'Report a nouveau','part_societe_consolidante':'Part de la societe consolidante','part_minoritaires':'Part des minoritaires',
   'total_capitaux_propres':'TOTAL I','emprunts_dettes_financieres':'Emprunts et dettes financieres',
   'impots_differes_provisionnes':'Impots (differes et provisionnes)','autres_dettes_non_courantes':'Autres dettes non courantes',
   'provisions_produits_avance':'Provisions et produits constates d avance','total_passifs_non_courants':'TOTAL II',
   'fournisseurs_rattaches':'Fournisseurs et comptes rattaches','impots_passif':'Impots','autres_dettes':'Autres dettes',
   'tresorerie_passif':'Tresorerie Passif','total_passifs_courants':'TOTAL III','total_general_passif':'TOTAL PASSIF (I+II+III)'}},
 'TCR': {'cols': ['n_debit','n_credit','n1_debit','n1_credit'], 'postes': {
   'ventes_marchandises':'Ventes de marchandises','produits_fabriques':'Produits fabriques','prestations_services':'Prestations de services',
   'ventes_travaux':'Vente de travaux','produits_annexes':'Produits annexes','rabais_remises_ristournes_accordes':'Rabais remises ristournes accordes',
   'chiffre_affaires_net':'Chiffre d affaires net','production_stockee_destockee':'Production stockee ou destockee',
   'production_immobilisee':'Production immobilisee','subvention_exploitation':'Subventions d exploitation','production_exercice':'I-Production de l exercice',
   'achats_marchandises_vendues':'Achats de marchandises vendues','matieres_premieres':'Matieres premieres','autres_approvisionnements':'Autres approvisionnements',
   'variation_stocks':'Variations des stocks','achats_etudes_prestations':'Achats d etudes et de prestations de services','autres_consommations':'Autres consommations',
   'rabais_remises_obtenus_achats':'Rabais remises ristournes obtenus sur achats','sous_traitance_generale':'Sous-traitance generale','locations':'Locations',
   'entretien_reparations':'Entretien reparations et maintenance','primes_assurances':'Primes d assurances','personnel_exterieur':'Personnel exterieur a l entreprise',
   'remuneration_intermediaires':'Remuneration d intermediaires et honoraires','publicite':'Publicite','deplacements_missions':'Deplacements missions et receptions',
   'autres_services':'Autres services','rabais_remises_obtenus_services':'Rabais remises ristournes obtenus sur services exterieurs',
   'consommations_exercice':'II-Consommations de l exercice','valeur_ajoutee_exploitation':'III-Valeur ajoutee d exploitation',
   'charges_personnel':'Charges de personnel','impots_taxes_assimiles':'Impots et taxes et versements assimiles',
   'excedent_brut_exploitation':'IV-Excedent brut d exploitation','autres_produits_operationnels':'Autres produits operationnels',
   'autres_charges_operationnelles':'Autres charges operationnelles','dotations_amortissements':'Dotations aux amortissements','provisions':'Provision',
   'pertes_valeur':'Pertes de valeur','reprises_pertes_valeur_provisions':'Reprise sur pertes de valeur et provisions',
   'resultat_operationnel':'V-Resultat operationnel','produits_financiers':'Produits financiers','charges_financieres':'Charges financieres',
   'resultat_financier':'VI-Resultat financier','resultat_ordinaire':'VII-Resultat ordinaire',
   'elements_extraordinaires_produits':'Elements extraordinaires (produits)','elements_extraordinaires_charges':'Elements extraordinaires (charges)',
   'resultat_extraordinaire':'VIII-Resultat extraordinaire','impots_exigibles_resultats':'Impots exigibles sur resultats',
   'impots_differes_resultats':'Impots differes (variations) sur resultats','resultat_net_exercice':'IX-RESULTAT NET DE L EXERCICE'}},
 'DECL': {'cols': ['valeur'], 'kinds': {'nif':'nif','raison_sociale':'texte','activite_principale':'texte','registre_commerce':'texte','adresse_siege':'texte','cac_cabinet':'texte','cac_nom':'texte','exercice_annee':'annee','annee_souscription':'annee'}, 'postes': {
   'nif':'NIF','raison_sociale':'Designation de l entreprise','activite_principale':'Activite','registre_commerce':'Registre de Commerce',
   'adresse_siege':'Adresse','cac_cabinet':'Cabinet CAC','cac_nom':'Nom CAC','exercice_annee':'Exercice - Annee',
   'annee_souscription':'Annee de souscription','chiffre_affaires_global_ht':'Chiffre d affaires global hors taxes',
   'resultat_comptable':'Resultat comptable','resultat_fiscal':'Resultat fiscal'}},
 'A1': {'cols': ['solde_debut','debit','credit','solde_fin'], 'postes': {'stocks_marchandises':'Stocks de marchandises','matieres_fournitures':'Matieres et fournitures','autres_approvisionnements':'Autres approvisionnements','encours_production_biens':'Encours de production de biens','encours_production_services':'Encours de production de services','stocks_produits':'Stocks de produits','stocks_provenant_immobilisations':'Stocks provenant d immobilisations','stocks_exterieur':'Stocks a l exterieur','total':'TOTAL'}},
 'A3': {'cols': ['montant'], 'postes': {'charges_locatives':'Charges locatives','etudes_recherches':'Etudes et recherches','documentation_divers':'Documentation et divers','transports_biens':'Transports de biens','frais_postaux':'Frais postaux','services_bancaires':'Services bancaires','cotisations_divers':'Cotisations et divers','total_autres_services':'TOTAL (1)','remunerations_personnel':'Remunerations du personnel','remuneration_exploitant':'Remuneration exploitant','cotisations_sociales':'Cotisations aux organismes sociaux','charges_sociales_exploitant':'Charges sociales exploitant','autres_charges_sociales':'Autres charges sociales','autres_charges_personnel':'Autres charges de personnels','total_charges_personnel':'TOTAL (2)','impots_sur_remunerations':'Impots sur remunerations','impots_non_recuperables':'Impots non recuperables','autres_impots_taxes':'Autres impots et taxes','total_impots':'TOTAL (3)','total_general':'TOTAL (1)+(2)+(3)'}},
 'A4': {'cols': ['montant'], 'postes': {'redevances_concessions_charges':'Redevances concessions (charges)','moins_values_sorties_actifs':'Moins values sorties actifs','jetons_presence_charges':'Jetons de presence (charges)','pertes_creances_irrecouvrables':'Perte sur creances irrecouvrables','quote_part_operations_commun_charges':'Quote-part operations en commun (charges)','amendes_penalites_dons':'Amendes penalites dons','charges_exceptionnelles_gestion':'Charges exceptionnelles gestion','autres_charges_gestion':'Autres charges gestion','total_charges':'TOTAL charges','redevances_concessions_produits':'Redevances concessions (produits)','plus_values_sorties_actifs':'Plus values sorties actifs','jetons_presence_produits':'Jetons de presence (produits)','quotes_parts_subventions_virees':'Quotes-parts subventions virees','quote_part_operations_commun_produits':'Quote-part operations en commun (produits)','rentrees_creances_amorties':'Rentree sur creances amorties','produits_exceptionnels_gestion':'Produits exceptionnels gestion','autres_produits_gestion':'Autres produits gestion','total_produits':'TOTAL produits'}},
 'A5': {'cols': ['dotations_cumulees_debut','dotations_exercice','diminutions_elements_sortis','dotations_cumulees_fin','dotations_fiscales_exercice','ecarts'], 'postes': {'goodwill':'Goodwill','immobilisations_incorporelles':'Immobilisations incorporelles','immobilisations_corporelles':'Immobilisations corporelles','participations':'Participations','autres_actifs_financiers_non_courants':'Autres actifs financiers non courants','total':'TOTAL'}},
 'A6': {'cols': ['montants_bruts','tva_deduite','montant_net_a_amortir'], 'postes': {'goodwill':'Goodwill','immobilisations_incorporelles':'Immobilisations incorporelles','immobilisations_corporelles':'Immobilisations corporelles','participations':'Participations','autres_actifs_financiers_non_courants':'Autres actifs financiers non courants','total':'TOTAL'}},
 'A7': {'dynamic': True, 'str_cols': ['date_acquisition'], 'cols': ['date_acquisition','montant_net_actif','amortissements_pratiques','valeur_nette_comptable','prix_cession','plus_value','moins_value'], 'postes': {}},
 'A8': {'cols': ['provisions_cumulees_debut','dotations_exercice','reprises_exercice','provisions_cumulees_fin'], 'postes': {'pertes_valeur_stocks':'Pertes valeurs stocks','pertes_valeur_creances':'Pertes valeurs creances','pertes_valeur_actions':'Pertes valeurs actions','provisions_pensions':'Provisions pensions','provisions_litiges':'Provisions litiges','autres_provisions_personnel':'Autres provisions personnel','provisions_impots':'Provisions impots','autres_provisions':'Autres provisions','total':'TOTAL'}},
 'A81': {'dynamic': True, 'cols': ['valeur_creance','perte_valeur_constituee'], 'postes': {}},
 'A82': {'dynamic': True, 'cols': ['valeur_nominale_debut','perte_valeur_constituee','valeur_nette_comptable'], 'postes': {}},
 'A9': {'cols': ['montant'], 'postes': {'resultat_net_benefice':'Resultat net Benefice','resultat_net_perte':'Resultat net Perte','charges_immeubles_non_affectes':'Charges immeubles non affectes','quote_part_cadeaux_publicitaires':'Quote-part cadeaux publicitaires','quote_part_sponsoring':'Quote-part sponsoring','frais_reception':'Frais reception','cotisations_dons':'Cotisations dons','impots_taxes_non_deductibles':'Impots taxes non deductibles','provisions_non_deductibles':'Provisions non deductibles','amortissements_non_deductibles':'Amortissements non deductibles','quote_part_frais_rd':'Quote-part frais RD','amortissements_credit_bail_preneur':'Amortissements credit bail preneur','loyers_hors_produits_financiers_bailleur':'Loyers hors produits financiers bailleur','ibs_impot_exigible':'IBS impot exigible','ibs_impot_differe':'IBS impot differe','pertes_valeur_non_deductibles':'Pertes valeurs non deductibles','amendes_penalites':'Amendes penalites','autres_reintegrations':'Autres reintegrations','total_reintegrations':'Total reintegrations','plus_values_cession_actif_immobilise':'Plus values cession actif','produits_plus_values_actions_bourse':'Produits plus values actions','revenus_distribution_benefices':'Revenus distribution benefices','amortissements_credit_bail_bailleur':'Amortissements credit bail bailleur','loyers_hors_charges_financieres_preneur':'Loyers hors charges financieres preneur','complement_amortissements':'Complement amortissements','autres_deductions':'Autres deductions','total_deductions':'Total deductions','total_deficits_a_deduire':'Total deficits a deduire','resultat_fiscal_benefice':'Resultat fiscal Benefice','resultat_fiscal_deficit':'Resultat fiscal Deficit'}},
 'A10': {'cols': ['montant'], 'postes': {'origine_report_a_nouveau_n1':'Report a nouveau N-1','origine_resultat_n1':'Resultat exercice N-1','origine_prelevements_reserves':'Prelevements sur reserves','origine_total':'TOTAL origine','affectation_reserves':'Reserves','affectation_augmentation_capital':'Augmentation capital','affectation_dividendes':'Dividendes','affectation_report_a_nouveau':'Report a nouveau','affectation_total':'TOTAL affectation'}},
 'A11': {'dynamic': True, 'cols': ['capitaux_propres','dont_capital','quote_part_capital_pct','resultat_dernier_exercice','prets_avances','dividendes_encaisses','valeur_comptable_titres'], 'postes': {}},
 'A12': {'dynamic': True, 'str_cols': ['nif','adresse'], 'cols': ['nif','adresse','montant_percu'], 'postes': {}},
 'A13': {'dynamic': True, 'cols': ['ca_imposable','ca_exonere','tap_acquittee'], 'postes': {}}
}
ANNEXE_NUM_MAP = {1:'A1',3:'A3',4:'A4',5:'A5',6:'A6',7:'A7',8:'A8',9:'A9',10:'A10',11:'A11',12:'A12',13:'A13'}
print('✅ Schémas V16 OK —', len(SCHEMAS), 'tableaux')

In [ ]:
# CLASSIFICATION PAR CONTENU (V13)
TITLE_SIGNATURES = [
 ('A81',['RELEVE DES PERTES DE VALEURS SUR CREANC'],[]), ('A82',['RELEVE DES PERTES DE VALEURS SUR ACTIONS'],[]),
 ('A1',['MOUVEMENTS DES STOCKS'],[]), ('A3',['CHARGES DE PERSONNEL','VERSEMENTS ASSIMILES'],[]),
 ('A4',['AUTRES CHARGES ET PRODUITS OPERATIONNELS'],[]), ('A5',['AMORTISSEMENTS ET PERTES DE VALEURS'],[]),
 ('A6',['IMMOBILISATIONS CREEES OU ACQUISES'],[]), ('A7',['IMMOBILISATIONS CEDEES'],[]),
 ('A8',['PROVISIONS ET PERTES DE VALEURS'],['RELEVE DES PERTES']), ('A9',['DETERMINATION DU RESULTAT FISCAL'],[]),
 ('A10',['AFFECTATION DU RESULTAT ET DES RESERVES'],[]), ('A11',['TABLEAU DES PARTICIPATIONS'],[]),
 ('A12',['COMMISSIONS ET COURTAGES'],[]), ('A13',['TAXE SUR L ACTIVITE PROFESSIONNELLE'],[])]
MAIN_SIGNATURES = [('ACTIF',['BILAN','ACTIF'],['PASSIF']), ('PASSIF',['BILAN','PASSIF'],[]), ('TCR',['COMPTE DE RESULTAT'],[]), ('DECL',['DECLARATION'],[])]
def _norm(t):
    t = strip_accents(t or '').upper(); t = re.sub(r'[^A-Z0-9/ ]+',' ', t); return ' '.join(t.split())
_TITLE_SIGS = [(c,[_norm(m) for m in ms],[_norm(f) for f in fs]) for c,ms,fs in TITLE_SIGNATURES]
_MAIN_SIGS = [(c,[_norm(m) for m in ms],[_norm(f) for f in fs]) for c,ms,fs in MAIN_SIGNATURES]
def _match(t, sigs):
    out = []
    for c, musts, forb in sigs:
        if all(m in t for m in musts) and not any(f in t for f in forb):
            if c not in out: out.append(c)
    return out
def _num_seg(t):
    codes, nums = [], []
    for m in re.finditer(r'(?:^| )(1[0-3]|[0-9])/([0-9])?(?![0-9])', t):
        n = int(m.group(1)); s = m.group(2)
        c = ('A8'+s) if (n==8 and s in ('1','2')) else ANNEXE_NUM_MAP.get(n)
        if c and c not in codes: codes.append(c); nums.append(n)
    return codes, nums
def types_from_title(titre):
    brut = _norm(titre)
    if not brut: return [], []
    segs = [_norm(s) for s in re.split(r'\s\|\s', titre or '') if s.strip()] or [brut]
    types, nums = [], []
    for seg in segs:
        tx = _match(seg, _TITLE_SIGS)
        if tx:
            for c in tx:
                if c not in types: types.append(c)
        else:
            cs, ns = _num_seg(seg)
            for c in cs:
                if c not in types: types.append(c)
            nums += ns
    if ('A81' in types or 'A82' in types) and 'A8' in types and not any('PROVISIONS ET PERTES DE VALEURS' in s for s in segs):
        types.remove('A8')
    if not types:
        types = _match(brut, _MAIN_SIGS)
        if 'ACTIF' in types and 'PASSIF' in types:
            types = ['ACTIF'] if brut.find('ACTIF') < brut.find('PASSIF') else ['PASSIF']
    return types, nums
PROMPT_CLASSIF = ('Lis cette page scannee d une liasse fiscale algerienne Serie G. '
 'Reponds en JSON strict: {"titre": ..., "type": ...}. '
 'titre = tous les titres de tableaux separes par |. type parmi ACTIF/PASSIF/TCR/DECL/ANNEXE/AUTRE.')
RULES = ['REGLES: JSON valide uniquement, sans markdown.', 'Aucune valeur inventee. Absent/illisible: null.', 'Montants en nombres JSON sans separateurs.', 'Parentheses = negatif.', 'Textes en chaines telles quelles.']
def build_prompt(types):
    L = ['Lis cette page scannee d une liasse fiscale algerienne Serie G.', 'Extrais en JSON strict les tableaux suivants.']
    L.append('Structure: { type_page, entete: {entreprise, nif, exercice, exercice_du, exercice_au, adresse, activite}, puis une cle par code.}')
    for t in types:
        spec = SCHEMAS[t]
        L.append('--- Code ' + t + ' ---')
        if spec.get('dynamic'):
            L.append('Tableau libre: {lignes: [ {libelle_imprime, valeurs: {colonne: nombre ou texte ou null}} ]}')
            L.append('Colonnes: ' + ', '.join(spec['cols']))
        else:
            L.append('Retourne: {lignes: [ {row_code, libelle_imprime, valeurs: {colonne: nombre ou null}} ]}')
            L.append('Colonnes: ' + ', '.join(spec['cols']))
            L.append('Row_code autorises (y compris les SOUS-TOTAUX):')
            for k, lab in spec['postes'].items(): L.append('- ' + k + ' : ' + lab)
            L.append('Retourne seulement les row_code avec au moins une valeur non nulle.')
    L += RULES
    return chr(10).join(L)
print('✅ Classification V16 OK')

In [ ]:
def norm_str(v):
    if v is None or isinstance(v, bool): return None
    s = ' '.join(str(v).split())
    return s if s and s.lower() not in ('null','none','n/a','na','-') else None
def norm_montant(v):
    if v is None or isinstance(v, bool): return None
    if isinstance(v,(int,float)): return float(v)
    s = str(v).strip()
    if s.lower() in ('null','none','n/a','na','-'): return None
    neg = (s.startswith('(') and s.endswith(')')) or s.startswith('-')
    s = re.sub('[^0-9.,-]','', s)
    if not s: return None
    if s.count(',')==1 and '.' not in s: s = s.replace(',','.')
    elif ',' in s: s = s.replace(',','')
    elif s.count('.')>1: s = s.replace('.','')
    try: return -float(s) if neg else float(s)
    except Exception: return None
def norm_nif(v):
    s = norm_str(v); return re.sub('[^0-9]','', s) if s else None
def norm_annee4(v):
    m = re.findall(r'20[0-9]{2}', norm_str(v) or ''); return m[-1] if m else None
def norm_by_kind(v, kind):
    if kind=='texte': return norm_str(v)
    if kind=='nif': return norm_nif(v)
    if kind=='annee': return norm_annee4(v)
    return norm_montant(v)
def normalise_fixed(t, block):
    spec = SCHEMAS[t]; kinds = spec.get('kinds', {})
    lignes = (block or {}).get('lignes') or []
    l2k = {norm_key(v): k for k, v in spec['postes'].items()}
    row_map = {}
    for lg in lignes:
        if not isinstance(lg, dict): continue
        rc = norm_str(lg.get('row_code'))
        if rc not in spec['postes']:
            lab = norm_str(lg.get('libelle_imprime'))
            if lab and norm_key(lab) in l2k: rc = l2k[norm_key(lab)]
        if rc in spec['postes']: row_map[rc] = lg
    out = {}
    for key in spec['postes']:
        vals = (row_map.get(key) or {}).get('valeurs') or {}
        if not isinstance(vals, dict): vals = {}
        out[key] = {col: norm_by_kind(vals.get(col), kinds.get(key,'montant')) for col in spec['cols']}
    return out
def normalise_dynamic(t, block):
    spec = SCHEMAS[t]; str_cols = set(spec.get('str_cols', [])); out = []
    for lg in (block or {}).get('lignes') or []:
        if not isinstance(lg, dict): continue
        vals = lg.get('valeurs') or {}
        if not isinstance(vals, dict): vals = {}
        row = {'libelle_imprime': norm_str(lg.get('libelle_imprime'))}
        for col in spec['cols']:
            row[col] = norm_str(vals.get(col)) if col in str_cols else norm_montant(vals.get(col))
        if row['libelle_imprime'] or any(v is not None for k, v in row.items() if k != 'libelle_imprime'): out.append(row)
    return out
def normalise_table(t, block):
    if SCHEMAS[t].get('dynamic'): return normalise_dynamic(t, block)
    return normalise_fixed(t, block)
def normalise_entete(data):
    ent = data.get('entete') or {}
    if not isinstance(ent, dict): ent = {}
    return {'entreprise': norm_str(ent.get('entreprise')), 'nif': norm_nif(ent.get('nif')), 'exercice': norm_str(ent.get('exercice')), 'exercice_du': norm_str(ent.get('exercice_du')), 'exercice_au': norm_str(ent.get('exercice_au')), 'adresse': norm_str(ent.get('adresse')), 'activite': norm_str(ent.get('activite'))}
def count_values(obj):
    c = 0
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == 'brut': continue
            c += count_values(v)
    elif isinstance(obj, list): c = sum(count_values(v) for v in obj)
    elif isinstance(obj,(int,float)) and not isinstance(obj,bool): c = 1
    elif isinstance(obj,str) and obj: c = 1
    return c
print('✅ Normalisation OK')

In [ ]:
def build_blank_page(page, src):
    n = page['index']+1
    return {'page_id':'p'+str(n).zfill(3),'pdf_page_index':page['index'],'numero_page_scannee':n,'fichier_source':src,
      'classification':{'type_page':'BLANCHE','types':[],'page_blanche':True,'page_utile':False},
      'statut_extraction':{'statut':'BLANCHE','nb_champs_extraits':0},'entete_page':{},'donnees':{}}
def build_page_object(page, types, data, rep, src):
    n = page['index']+1; data = data if isinstance(data, dict) else {}
    donnees = {'brut': data}
    for t in types:
        donnees[t] = normalise_table(t, data.get(t))
    return {'page_id':'p'+str(n).zfill(3),'pdf_page_index':page['index'],'numero_page_scannee':n,'fichier_source':src,
      'classification':{'type_page':types[0] if types else 'AUTRE','types':types,'page_blanche':False,'page_utile':bool(types)},
      'statut_extraction':{'statut':'OK' if data else 'ECHEC','nb_champs_extraits':count_values(donnees)},
      'entete_page':normalise_entete(data),'donnees':donnees,
      'tokens_in':rep.get('tokens_in') if rep else 0,'tokens_out':rep.get('tokens_out') if rep else 0}
def merge_core(base, new):
    if new is None: return base
    if base is None: return copy.deepcopy(new)
    for k, cols in new.items():
        if k not in base: base[k] = copy.deepcopy(cols)
        elif isinstance(cols, dict):
            for col, val in cols.items():
                if base[k].get(col) is None and val is not None: base[k][col] = val
    return base
def build_synthese(pobjs):
    syn = {'actif':None,'passif':None,'tcr':None,'decl':None,'annexes':{}}
    for p in pobjs:
        types = p['classification'].get('types') or []; d = p.get('donnees', {})
        if 'ACTIF' in types and syn['actif'] is None: syn['actif'] = d.get('ACTIF')
        if 'PASSIF' in types and syn['passif'] is None: syn['passif'] = d.get('PASSIF')
        if 'TCR' in types: syn['tcr'] = merge_core(syn['tcr'], d.get('TCR'))
        if 'DECL' in types and syn['decl'] is None: syn['decl'] = d.get('DECL')
        for t in types:
            if t not in ('ACTIF','PASSIF','TCR','DECL','AUTRE') and syn['annexes'].get(t) is None: syn['annexes'][t] = d.get(t)
    return syn
def build_document(pdf_path, pages, pobjs, ti, to, elapsed):
    return {'schema_version':'16.0','type_document':'liasse_fiscale_algerienne_serie_g',
      'document':{'fichier_source':pdf_path.name,'nb_pages_pdf':len(pages),
        'nb_pages_blanches':sum(1 for p in pobjs if p['classification']['type_page']=='BLANCHE'),
        'date_extraction':datetime.now().strftime('%Y-%m-%d %H:%M:%S'),'duree_s':round(elapsed,2),
        'tokens_in':ti,'tokens_out':to,'tokens_total':ti+to},
      'pages':pobjs,'synthese':build_synthese(pobjs)}
print('✅ Construction pages OK')

In [ ]:
# RÈGLES DE CALCUL V16 — sous-totaux ACTIF ajoutés
COLS_ACTIF = ['montant_brut','amortissements_provisions_pertes','net_n','net_n1']
COLS_PASSIF = ['n','n1']
FORMULES = [
 # Sous-totaux ACTIF (V16)
 ('ACTIF','immobilisations_corporelles',COLS_ACTIF,[('+','terrains'),('+','batiments'),('+','autres_immobilisations_corporelles'),('+','immobilisations_en_concession')]),
 ('ACTIF','immobilisations_financieres',COLS_ACTIF,[('+','titres_mis_en_equivalence'),('+','autres_participations_creances'),('+','autres_titres_immobilises'),('+','prets_actifs_financiers_non_courants'),('+','impots_differes_actif')]),
 ('ACTIF','creances_et_emplois_assimiles',COLS_ACTIF,[('+','clients'),('+','autres_debiteurs'),('+','impots_assimiles_actif'),('+','autres_creances_assimiles')]),
 ('ACTIF','disponibilites_et_assimiles',COLS_ACTIF,[('+','placements_financiers_courants'),('+','tresorerie_actif')]),
 ('ACTIF','total_actif_non_courant',COLS_ACTIF,[('+','ecarts_acquisition_goodwill'),('+','immobilisations_incorporelles'),('+','immobilisations_corporelles'),('+','immobilisations_en_cours'),('+','immobilisations_financieres')]),
 ('ACTIF','total_actif_courant',COLS_ACTIF,[('+','stocks_encours'),('+','creances_et_emplois_assimiles'),('+','disponibilites_et_assimiles')]),
 ('ACTIF','total_general_actif',COLS_ACTIF,[('+','total_actif_non_courant'),('+','total_actif_courant')]),
 ('PASSIF','total_capitaux_propres',COLS_PASSIF,[('+','capital_emis'),('+','capital_non_appele'),('+','primes_reserves'),('+','ecart_reevaluation'),('+','ecart_equivalence'),('+','resultat_net_passif'),('+','report_a_nouveau'),('+','part_societe_consolidante'),('+','part_minoritaires')]),
 ('PASSIF','total_passifs_non_courants',COLS_PASSIF,[('+','emprunts_dettes_financieres'),('+','impots_differes_provisionnes'),('+','autres_dettes_non_courantes'),('+','provisions_produits_avance')]),
 ('PASSIF','total_passifs_courants',COLS_PASSIF,[('+','fournisseurs_rattaches'),('+','impots_passif'),('+','autres_dettes'),('+','tresorerie_passif')]),
 ('PASSIF','total_general_passif',COLS_PASSIF,[('+','total_capitaux_propres'),('+','total_passifs_non_courants'),('+','total_passifs_courants')]),
 ('TCR','chiffre_affaires_net','NET',[('+','ventes_marchandises'),('+','produits_fabriques'),('+','prestations_services'),('+','ventes_travaux'),('+','produits_annexes'),('+','rabais_remises_ristournes_accordes')]),
 ('TCR','production_exercice','NET',[('+','chiffre_affaires_net'),('+','production_stockee_destockee'),('+','production_immobilisee'),('+','subvention_exploitation')]),
 ('TCR','consommations_exercice','NET',[('+','achats_marchandises_vendues'),('+','matieres_premieres'),('+','autres_approvisionnements'),('+','variation_stocks'),('+','achats_etudes_prestations'),('+','autres_consommations'),('+','sous_traitance_generale'),('+','locations'),('+','entretien_reparations'),('+','primes_assurances'),('+','personnel_exterieur'),('+','remuneration_intermediaires'),('+','publicite'),('+','deplacements_missions'),('+','autres_services')]),
 ('TCR','valeur_ajoutee_exploitation','NET',[('+','production_exercice'),('+','consommations_exercice')]),
 ('TCR','excedent_brut_exploitation','NET',[('+','valeur_ajoutee_exploitation'),('+','charges_personnel'),('+','impots_taxes_assimiles')]),
 ('TCR','resultat_operationnel','NET',[('+','excedent_brut_exploitation'),('+','autres_produits_operationnels'),('+','autres_charges_operationnelles'),('+','dotations_amortissements'),('+','provisions'),('+','pertes_valeur'),('+','reprises_pertes_valeur_provisions')]),
 ('TCR','resultat_financier','NET',[('+','produits_financiers'),('+','charges_financieres')]),
 ('TCR','resultat_ordinaire','NET',[('+','resultat_operationnel'),('+','resultat_financier')]),
 ('TCR','resultat_extraordinaire','NET',[('+','elements_extraordinaires_produits'),('+','elements_extraordinaires_charges')]),
 ('TCR','resultat_net_exercice','NET',[('+','resultat_ordinaire'),('+','resultat_extraordinaire'),('+','impots_exigibles_resultats'),('+','impots_differes_resultats')]),
 ('A1','total',['solde_debut','debit','credit','solde_fin'],[('+','stocks_marchandises'),('+','matieres_fournitures'),('+','autres_approvisionnements'),('+','encours_production_biens'),('+','encours_production_services'),('+','stocks_produits'),('+','stocks_provenant_immobilisations'),('+','stocks_exterieur')]),
 ('A5','total',['dotations_cumulees_debut','dotations_exercice','diminutions_elements_sortis','dotations_cumulees_fin','dotations_fiscales_exercice','ecarts'],[('+','goodwill'),('+','immobilisations_incorporelles'),('+','immobilisations_corporelles'),('+','participations'),('+','autres_actifs_financiers_non_courants')]),
 ('A8','total',['provisions_cumulees_debut','dotations_exercice','reprises_exercice','provisions_cumulees_fin'],[('+','pertes_valeur_stocks'),('+','pertes_valeur_creances'),('+','pertes_valeur_actions'),('+','provisions_pensions'),('+','provisions_litiges'),('+','autres_provisions_personnel'),('+','provisions_impots'),('+','autres_provisions')]),
 ('A9','total_reintegrations',['montant'],[('+','charges_immeubles_non_affectes'),('+','quote_part_cadeaux_publicitaires'),('+','quote_part_sponsoring'),('+','frais_reception'),('+','cotisations_dons'),('+','impots_taxes_non_deductibles'),('+','provisions_non_deductibles'),('+','amortissements_non_deductibles'),('+','amendes_penalites'),('+','autres_reintegrations')]),
 ('A9','total_deductions',['montant'],[('+','plus_values_cession_actif_immobilise'),('+','produits_plus_values_actions_bourse'),('+','revenus_distribution_benefices'),('+','complement_amortissements'),('+','autres_deductions')]),
 ('A10','origine_total',['montant'],[('+','origine_report_a_nouveau_n1'),('+','origine_resultat_n1'),('+','origine_prelevements_reserves')]),
 ('A10','affectation_total',['montant'],[('+','affectation_reserves'),('+','affectation_augmentation_capital'),('+','affectation_dividendes'),('+','affectation_report_a_nouveau')])
]
FORMULES_LIGNE = [
 ('ACTIF','net_n',[('+','montant_brut'),('-','amortissements_provisions_pertes')]),
 ('A1','solde_fin',[('+','solde_debut'),('+','debit'),('-','credit')]),
 ('A5','dotations_cumulees_fin',[('+','dotations_cumulees_debut'),('+','dotations_exercice'),('-','diminutions_elements_sortis')]),
 ('A8','provisions_cumulees_fin',[('+','provisions_cumulees_debut'),('+','dotations_exercice'),('-','reprises_exercice')]),
 ('A7','valeur_nette_comptable',[('+','montant_net_actif'),('-','amortissements_pratiques')]),
 ('A82','valeur_nette_comptable',[('+','valeur_nominale_debut'),('-','perte_valeur_constituee')])
]
CONTROLES = [
 ('Equilibre du bilan (N)','critique',[('+','ACTIF','total_general_actif','net_n')],[('+','PASSIF','total_general_passif','n')]),
 ('Equilibre du bilan (N-1)','critique',[('+','ACTIF','total_general_actif','net_n1')],[('+','PASSIF','total_general_passif','n1')]),
 ('Resultat net : TCR = passif (N)','critique',[('+','TCR','resultat_net_exercice','NET')],[('+','PASSIF','resultat_net_passif','n')]),
 ('Resultat net : TCR = ligne I tableau 9','critique',[('+','TCR','resultat_net_exercice','NET')],[('+','A9','resultat_net_benefice','montant'),('-','A9','resultat_net_perte','montant')]),
 ('Stocks : total A1 (fin) = stocks bilan (N)','majeur',[('+','A1','total','solde_fin')],[('+','ACTIF','stocks_encours','net_n')]),
 ('Amortissements : cumul fin A5 = colonne amort. bilan','majeur',[('+','A5','total','dotations_cumulees_fin')],[('+','ACTIF','total_actif_non_courant','amortissements_provisions_pertes')]),
 ('Fiscal : resultat = I + reint - ded - deficits','critique',[('+','A9','resultat_fiscal_benefice','montant'),('-','A9','resultat_fiscal_deficit','montant')],[('+','A9','resultat_net_benefice','montant'),('-','A9','resultat_net_perte','montant'),('+','A9','total_reintegrations','montant'),('-','A9','total_deductions','montant'),('-','A9','total_deficits_a_deduire','montant')]),
 ('Affectation : origine = affectation','critique',[('+','A10','origine_total','montant')],[('+','A10','affectation_total','montant')])
]
print('✅ Règles V16 OK —', len(FORMULES), 'formules,', len(CONTROLES), 'contrôles')

In [ ]:
# MOTEUR DE COHÉRENCE — écart extrait vs recalculé
TOLERANCE_DA = 1.0
def lire_bloc(bloc):
    if isinstance(bloc, list):
        return [('ligne_'+str(i).zfill(3), {k:v for k,v in row.items() if k!='libelle_imprime'}, row.get('libelle_imprime')) for i, row in enumerate(bloc) if isinstance(row, dict)]
    if isinstance(bloc, dict):
        if isinstance(bloc.get('lignes'), list):
            return [(lg.get('row_code') or ('ligne_'+str(i).zfill(3)), lg.get('valeurs') or {}, lg.get('libelle_imprime')) for i, lg in enumerate(bloc['lignes']) if isinstance(lg, dict)]
        return [(rc, vals, None) for rc, vals in bloc.items() if isinstance(vals, dict)]
    return []
def iter_blocs(doc):
    for page in doc.get('pages', []):
        for tab, bloc in (page.get('donnees') or {}).items():
            if tab in ('brut','AUTRE') or bloc in (None, {}, []): continue
            yield page, tab, bloc
def _index_postes(doc):
    idx, dyn = {}, {}
    for _p, tab, bloc in iter_blocs(doc):
        for cle, vals, _lib in lire_bloc(bloc):
            rens = {k:v for k, v in (vals or {}).items() if v is not None}
            if rens: idx.setdefault((tab, cle), {}).update(rens)
            if cle != 'total':
                for col, v in rens.items():
                    if isinstance(v,(int,float)) and not isinstance(v,bool): dyn[(tab,col)] = dyn.get((tab,col),0.0)+float(v)
    for (tab,col), s in dyn.items(): idx.setdefault((tab,' TOTAL '), {})[col] = s
    return idx
def _valeur(idx, tab, rc, col):
    vals = idx.get((tab, rc))
    if vals is None: return None
    if col in ('NET','NET_N1'):
        suf = ('n_credit','n_debit') if col=='NET' else ('n1_credit','n1_debit')
        c, d = vals.get(suf[0]), vals.get(suf[1])
        if c is None and d is None: return None
        return float(c or 0) - float(d or 0)
    v = vals.get(col)
    return float(v) if isinstance(v,(int,float)) and not isinstance(v,bool) else None
def _somme(idx, termes):
    tot, vus = 0.0, 0
    for signe, tab, rc, col in termes:
        v = _valeur(idx, tab, rc, col)
        if v is not None: tot += v if signe=='+' else -v; vus += 1
    return tot if vus else None
def _statut(ecart, tol):
    if abs(ecart) <= 1e-9: return 'coherent'
    if abs(ecart) <= tol: return 'coherent_arrondi'
    return 'ecart_significatif'
def verifier_coherence(doc, tol=TOLERANCE_DA):
    idx = _index_postes(doc); rf, rc_ = [], []
    for tab, cible, cols, comps in FORMULES:
        colonnes = ['NET','NET_N1'] if cols=='NET' else cols
        for col in colonnes:
            dec = _valeur(idx, tab, cible, col); rec = _somme(idx, [(s,tab,r,col) for s,r in comps])
            if dec is None or rec is None: continue   # sous-total absent -> pas d'écart inventé
            e = dec - rec
            rf.append({'type':'agregat','tableau':tab,'poste':cible,'colonne':col,'valeur_extraite':dec,'valeur_recalculee':rec,'ecart':round(e,2),'statut':_statut(e,tol)})
    for tab, col_cible, comps in FORMULES_LIGNE:
        for (t, rc) in list(idx.keys()):
            if t != tab or rc==' TOTAL ': continue
            dec = _valeur(idx, tab, rc, col_cible); rec = _somme(idx, [(s,tab,rc,c) for s,c in comps])
            if dec is None or rec is None: continue
            e = dec - rec
            rf.append({'type':'ligne','tableau':tab,'poste':rc,'colonne':col_cible,'valeur_extraite':dec,'valeur_recalculee':rec,'ecart':round(e,2),'statut':_statut(e,tol)})
    for lib, grav, g, d in CONTROLES:
        a, b = _somme(idx, g), _somme(idx, d)
        if a is None or b is None:
            rc_.append({'controle':lib,'gravite':grav,'statut':'non_verifiable','valeur_a':a,'valeur_b':b,'ecart':None}); continue
        e = a - b
        rc_.append({'controle':lib,'gravite':grav,'valeur_a':round(a,2),'valeur_b':round(b,2),'ecart':round(e,2),'statut':_statut(e,tol)})
    ec_f = [f for f in rf if f['statut']=='ecart_significatif']; ec_c = [c for c in rc_ if c['statut']=='ecart_significatif']
    crit = [c for c in ec_c if c['gravite']=='critique']
    return {'tolerance_da':tol,'formules':rf,'controles_croises':rc_,
      'synthese':{'formules_verifiees':len(rf),'formules_en_ecart':len(ec_f),'controles_verifies':len([c for c in rc_ if c['statut']!='non_verifiable']),'controles_en_ecart':len(ec_c),'ecarts_critiques':len(crit),'liasse_exploitable':not crit}}
def enrichir_lignes(doc, rapport):
    index = {(f['tableau'], f['poste'], f['colonne']): f for f in rapport['formules']}
    for page, tab, bloc in iter_blocs(doc):
        cible = page.setdefault('controles_donnees', {}).setdefault(tab, {})
        for cle, vals, _lib in lire_bloc(bloc):
            ctrl = {}
            for col, val in (vals or {}).items():
                if val is None or not isinstance(val,(int,float)) or isinstance(val,bool): continue
                f = index.get((tab, cle, col))
                ctrl[col] = ({'valeur_certaine': f['statut']!='ecart_significatif','valeur_extraite':f['valeur_extraite'],'valeur_recalculee':f['valeur_recalculee'],'ecart':f['ecart'],'statut':f['statut']} if f else {'valeur_certaine':None,'statut':'non_verifiable'})
            if ctrl: cible[cle] = ctrl
        if not cible: page['controles_donnees'].pop(tab, None)
    return doc
def appliquer_controles(doc):
    rapport = verifier_coherence(doc)
    enrichir_lignes(doc, rapport)
    doc['controles'] = rapport; doc['schema_version'] = '16.0'
    return doc
print('✅ Moteur de cohérence V16 OK')

In [ ]:
# MAPPING EXCEL V16 — résolution des lignes par libellé normalisé (robuste)
MAP_FEUILLES = {'ACTIF':'1. Bilan Actif','PASSIF':'2. Bilan Passif','TCR':'3. TCR','A1':'4. Stocks','A3':'5. Charges & produits','A4':'5. Charges & produits','A5':'6. Amort. & Immo.','A6':'6. Amort. & Immo.','A7':'7. Cessions & Provisions','A8':'7. Cessions & Provisions','A81':'8. Pertes de valeurs','A82':'8. Pertes de valeurs','A9':'9. Résultat fiscal','A10':'10. Affectation & Particip.','A11':'10. Affectation & Particip.','A12':'11. Commissions & TAP','A13':'11. Commissions & TAP'}
MAP_COLONNES = {'ACTIF':{'montant_brut':'B','amortissements_provisions_pertes':'C','net_n':'D','net_n1':'E'},'PASSIF':{'n':'B','n1':'C'},'TCR':{'n_debit':'B','n_credit':'C','n1_debit':'D','n1_credit':'E'},'A1':{'solde_debut':'B','debit':'C','credit':'D','solde_fin':'E'},'A3':{'montant':'B'},'A4':{'montant':'B'},'A5':{'dotations_cumulees_debut':'B','dotations_exercice':'C','diminutions_elements_sortis':'D','dotations_cumulees_fin':'E','dotations_fiscales_exercice':'F','ecarts':'G'},'A6':{'montants_bruts':'B','tva_deduite':'C','montant_net_a_amortir':'D'},'A7':{'date_acquisition':'B','montant_net_actif':'C','amortissements_pratiques':'D','valeur_nette_comptable':'E','prix_cession':'F','plus_value':'G','moins_value':'H'},'A8':{'provisions_cumulees_debut':'B','dotations_exercice':'C','reprises_exercice':'D','provisions_cumulees_fin':'E'},'A81':{'valeur_creance':'B','perte_valeur_constituee':'C'},'A82':{'valeur_nominale_debut':'B','perte_valeur_constituee':'C','valeur_nette_comptable':'D'},'A9':{'montant':'B'},'A10':{'montant':'B'},'A11':{'capitaux_propres':'B','dont_capital':'C','quote_part_capital_pct':'D','resultat_dernier_exercice':'E','prets_avances':'F','dividendes_encaisses':'G','valeur_comptable_titres':'H'},'A12':{'nif':'B','adresse':'C','montant_percu':'D'},'A13':{'ca_imposable':'B','ca_exonere':'C','tap_acquittee':'D'}}
MAP_DYNAMIQUES = {'A7':(11,20,'A'),'A81':(10,23,'A'),'A82':(29,42,'A'),'A11':(26,34,'A'),'A12':(10,24,'A'),'A13':(30,41,'A')}
MAP_ENTETE = {'nif':'B4','entreprise':'B5','exercice':'B6'}
MAP_MILLESIMES = [('1. Bilan Actif','B8','N'),('1. Bilan Actif','E8','N1'),('2. Bilan Passif','B8','N'),('2. Bilan Passif','C8','N1'),('3. TCR','B8','N'),('3. TCR','D8','N1')]
def _index_lignes(wb):
    idx = {}
    for feuille in wb.sheetnames:
        ws = wb[feuille]; labels = []
        for r in range(1, ws.max_row+1):
            v = ws.cell(row=r, column=1).value
            if isinstance(v, str) and v.strip(): labels.append((r, norm_key(v)))
        idx[feuille] = labels
    return idx
def _trouver_ligne(idx, feuille, label):
    cible = norm_key(label)
    if not cible: return None
    for r, lab in idx.get(feuille, []):
        if lab == cible: return r
    best = None
    for r, lab in idx.get(feuille, []):
        if cible in lab:
            if best is None or len(lab) < len(best[1]): best = (r, lab)
    return best[0] if best else None
print('✅ Mapping V16 OK')

In [ ]:
# TRANSPOSITION V16 — sous-totaux = formules (jamais écrites), détails écrits, écarts annotés
_ROUGE = Border(*[Side(style='medium', color='FF0000')]*4)
_ROUGE_FOND = PatternFill('solid', start_color='FFC7CE')
_ROUGE_TEXTE = Font(bold=True, color='9C0006')
_LIBRE = Protection(locked=False); _VERROU = Protection(locked=True)
def _est_formule(c): return isinstance(c.value, str) and c.value.startswith('=')
def _postes_du_document(doc):
    fixes, dyn = [], {}
    for _p, tab, bloc in iter_blocs(doc):
        for cle, vals, libelle in lire_bloc(bloc):
            vals = vals or {}
            est_libre = str(cle).startswith('ligne_')
            if not est_libre and tab in SCHEMAS and not SCHEMAS[tab].get('dynamic') and cle in SCHEMAS[tab]['postes']:
                for col, v in vals.items():
                    if v is not None: fixes.append((tab, cle, col, v))
            elif tab in MAP_DYNAMIQUES:
                if any(v is not None for v in vals.values()) or libelle: dyn.setdefault(tab, []).append({'libelle':libelle,'valeurs':vals})
    return fixes, dyn
def _ecrire_valeurs(wb, doc, idx_lignes):
    fixes, dyn = _postes_du_document(doc)
    ecrits, totaux, ignores = 0, [], []
    ident = doc.get('identite') or {}
    ws1 = wb[MAP_FEUILLES['ACTIF']]
    if ident.get('nif_15'): ws1[MAP_ENTETE['nif']] = ident['nif_15']
    if ident.get('entreprise'): ws1[MAP_ENTETE['entreprise']] = ident['entreprise']
    if ident.get('exercice_clos'): ws1[MAP_ENTETE['exercice']] = ident['exercice_clos']
    for feuille, coord, quel in MAP_MILLESIMES:
        an = ident.get('annee_n') if quel=='N' else ident.get('annee_n1')
        if an: wb[feuille][coord] = ('N : '+str(an)) if quel=='N' else ('N-1 : '+str(an))
    for tab, rc, col, val in fixes:
        feuille = MAP_FEUILLES.get(tab); colx = MAP_COLONNES.get(tab, {}).get(col)
        if not feuille or not colx: ignores.append((tab, rc, col)); continue
        row = _trouver_ligne(idx_lignes, feuille, SCHEMAS[tab]['postes'][rc])
        if row is None: ignores.append((tab, rc, col)); continue
        cible = wb[feuille][colx + str(row)]
        if _est_formule(cible):   # sous-total / total = formule -> comparaison seulement
            if isinstance(val,(int,float)) and not isinstance(val,bool): totaux.append((feuille, colx+str(row), tab, rc, col, float(val)))
            continue
        cible.value = val; ecrits += 1
    for tab, lignes in dyn.items():
        debut, fin, col_lib = MAP_DYNAMIQUES[tab]; ws = wb[MAP_FEUILLES[tab]]; r = debut
        for item in lignes:
            if r > fin: ignores.append((tab, 'ligne_'+str(r), 'depassement')); break
            if col_lib and item.get('libelle'): ws[col_lib+str(r)] = item['libelle']
            for col, v in (item.get('valeurs') or {}).items():
                colx = MAP_COLONNES.get(tab, {}).get(col)
                if colx and v is not None:
                    cc = ws[colx+str(r)]
                    if _est_formule(cc):
                        if isinstance(v,(int,float)) and not isinstance(v,bool): totaux.append((MAP_FEUILLES[tab], colx+str(r), tab, 'ligne_'+str(r), col, float(v)))
                    else: cc.value = v; ecrits += 1
            r += 1
    return ecrits, totaux, ignores
def _comparer_et_annoter(chemin_calcule, wb, totaux, doc=None, tol=TOLERANCE_DA):
    calc = None
    if chemin_calcule:
        try: calc = openpyxl.load_workbook(chemin_calcule, data_only=True)
        except Exception: calc = None
    replis = {}
    if calc is None and doc:
        for f in (doc.get('controles') or {}).get('formules', []): replis[(f['tableau'], f['poste'], f['colonne'])] = f['valeur_recalculee']
    anomalies = []
    for feuille, coord, tab, rc, col, extraite in totaux:
        if calc is not None:
            recalc = calc[feuille][coord].value
            if recalc is None: continue
            recalc = float(recalc)
        else:
            if (tab, rc, col) not in replis: continue
            recalc = float(replis[(tab, rc, col)])
        e = extraite - recalc
        if abs(e) <= tol: continue
        c = wb[feuille][coord]; c.border = _ROUGE; c.fill = _ROUGE_FOND; c.font = _ROUGE_TEXTE
        c.comment = Comment('INCOHERENCE\nValeur lue: '+format(extraite,',.2f')+'\nValeur recalculee: '+format(recalc,',.2f')+'\nEcart: '+format(e,',.2f'), 'Controle extraction')
        c.comment.width = 320; c.comment.height = 170
        anomalies.append({'tableau':tab,'poste':rc,'colonne':col,'feuille':feuille,'cellule':coord,'valeur_extraite':extraite,'valeur_recalculee':recalc,'ecart':round(e,2)})
    return anomalies
def _feuille_archive(wb, doc, anomalies, ignores):
    if '0. Données extraites' in wb.sheetnames: del wb['0. Données extraites']
    ws = wb.create_sheet('0. Données extraites', 0)
    ident = doc.get('identite') or {}; syn = (doc.get('controles') or {}).get('synthese', {})
    ws['A1'] = 'DONNEES EXTRAITES DE LA LIASSE — PIECE DE REFERENCE'; ws['A1'].font = Font(bold=True, size=14)
    infos = [('Fichier',doc.get('document',{}).get('fichier_source')),('Entreprise',ident.get('entreprise')),('NIF',ident.get('nif_15')),('Exercice',ident.get('exercice_clos')),('Formules en ecart',syn.get('formules_en_ecart')),('Controles en ecart',syn.get('controles_en_ecart')),('Ecarts critiques',syn.get('ecarts_critiques'))]
    r = 3
    for lib, val in infos: ws.cell(r,1,lib).font = Font(bold=True); ws.cell(r,2,val); r += 1
    r += 1; ws.cell(r,1,'ECARTS SOUS-TOTAUX / TOTAUX').font = Font(bold=True, size=12); r += 1
    for a in anomalies: ws.cell(r,1,a['tableau']); ws.cell(r,2,a['poste']); ws.cell(r,3,a['colonne']); ws.cell(r,4,a['feuille']+'!'+a['cellule']); ws.cell(r,5,a['valeur_extraite']); ws.cell(r,6,a['valeur_recalculee']); ws.cell(r,7,a['ecart']); r += 1
    if not anomalies: ws.cell(r,2,'Aucun écart au-delà de la tolérance.'); r += 1
    return ws
def _proteger_formules(wb):
    nl = nv = 0
    for ws in wb.worksheets:
        for ligne in ws.iter_rows():
            for c in ligne:
                if _est_formule(c): c.protection = _VERROU; nv += 1
                else: c.protection = _LIBRE; nl += 1
        ws.protection.sheet = True
    return nl, nv
def transposer(doc, sortie_xlsx, modele=None, tol=TOLERANCE_DA):
    modele = Path(modele or MODELE_XLSX); sortie = Path(sortie_xlsx); tmp = sortie.with_suffix('.tmp.xlsx')
    shutil.copy(modele, tmp)
    wb = openpyxl.load_workbook(tmp)
    for ws in wb.worksheets: ws.protection.sheet = False
    idx_lignes = _index_lignes(wb)
    ecrits, totaux, ignores = _ecrire_valeurs(wb, doc, idx_lignes)
    wb.save(tmp)
    recalcule = False
    if RECALC_DISPONIBLE:
        try: subprocess.run([sys.executable, str(RECALC_TOOL), str(tmp), '180'], capture_output=True, timeout=300); recalcule = True
        except Exception as e: log('Recalcul indisponible: '+str(e))
    wb = openpyxl.load_workbook(tmp)
    anomalies = _comparer_et_annoter(tmp if recalcule else None, wb, totaux, doc, tol)
    _feuille_archive(wb, doc, anomalies, ignores)
    _proteger_formules(wb)
    wb.save(sortie); tmp.unlink(missing_ok=True)
    if RECALC_DISPONIBLE:
        try: subprocess.run([sys.executable, str(RECALC_TOOL), str(sortie), '180'], capture_output=True, timeout=300)
        except Exception as e: log('Recalcul final indisponible: '+str(e))
    doc.setdefault('controles', {})['transposition'] = {'classeur':sortie.name,'valeurs_ecrites':ecrits,'totaux_compares':len(totaux),'ecarts_signales':len(anomalies),'postes_non_reportes':len(ignores),'detail_ecarts':anomalies}
    return doc['controles']['transposition']
print('✅ Transposition V16 OK')

In [ ]:
# PIPELINE + EXÉCUTION V16
deja = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter = [p for p in pdfs if p.stem not in deja]
log('A traiter: '+str(len(a_traiter)))
prompt_cache = {}
for num, pdf_path in enumerate(a_traiter, start=1):
    t0 = time.time()
    try:
        pages = pdf_to_pages(pdf_path)
        pobjs, actives = [], []
        for p in pages:
            (pobjs.append(build_blank_page(p, pdf_path.name)) if is_blank(p['image']) else actives.append(p))
        ti = to = 0
        if actives:
            mini = [resize(p['image'], CLASSIF_RES) for p in actives]
            reps1 = []
            for bs in chunks(mini, CLASSIF_BATCH_SIZE): reps1 += ask_batch(PROMPT_CLASSIF, bs)
            ti += sum(r['tokens_in'] for r in reps1); to += sum(r['tokens_out'] for r in reps1)
            ptyped = []
            for p, rep in zip(actives, reps1):
                d = parse_json(rep['text']); types, nums = types_from_title(norm_str(d.get('titre')))
                if not types:
                    tm = (norm_str(d.get('type')) or '').upper()
                    types = [tm] if tm in ('ACTIF','PASSIF','TCR','DECL') else ['AUTRE']
                p['types'] = types; ptyped.append((p, tuple(types)))
            groups = {}
            for p, key in ptyped: groups.setdefault(key, []).append(p)
            extracted = {}
            for key, plist in groups.items():
                prompt = prompt_cache.get(key)
                if prompt is None:
                    prompt = build_prompt(list(key)); prompt_cache[key] = prompt
                for batch in chunks(plist, GPU_BATCH_SIZE):
                    reps = ask_batch(prompt, [p['image'] for p in batch])
                    for p, rep in zip(batch, reps):
                        ti += rep['tokens_in']; to += rep['tokens_out']
                        extracted[p['index']] = build_page_object(p, list(key), parse_json(rep['text']), rep, pdf_path.name)
                    gc.collect(); torch.cuda.empty_cache()
            for p in actives:
                pobjs.append(extracted.get(p['index'], build_page_object(p, ['AUTRE'], {}, None, pdf_path.name)))
        pobjs.sort(key=lambda x: x['pdf_page_index'])
        doc = build_document(pdf_path, pages, pobjs, ti, to, time.time()-t0)
        doc = appliquer_controles(doc)
        with open(JSON_DIR / (pdf_path.stem+'.json'), 'w', encoding='utf-8') as f: json.dump(doc, f, ensure_ascii=False, indent=2, default=str)
        if MODELE_XLSX.exists():
            try: transposer(doc, XLSX_DIR / (pdf_path.stem+'.xlsx'))
            except Exception as e: log('❌ Transposition '+pdf_path.name+' : '+str(e))
        (CONTROLE_DIR / (pdf_path.stem+'.controle.json')).write_text(json.dumps(doc, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
        log('['+str(num).zfill(4)+'] ✅ '+pdf_path.name+' | '+str(round(time.time()-t0,1))+'s')
    except Exception as e:
        log('['+str(num).zfill(4)+'] ❌ '+pdf_path.name+' — '+str(e))
log('✅ V16 terminé')

In [ ]:
# VALIDATION — sous-totaux + écarts
files = sorted(JSON_DIR.glob('*.json'))
if files:
    d = json.load(open(files[-1], encoding='utf-8'))
    a = (d.get('synthese') or {}).get('actif') or {}
    for k in ('immobilisations_corporelles','immobilisations_financieres','creances_et_emplois_assimiles','disponibilites_et_assimiles'):
        print(k, '->', a.get(k))
    for f in (d.get('controles') or {}).get('formules', []):
        if f['statut'] == 'ecart_significatif': print('ECART', f['tableau'], f['poste'], f['colonne'], f['ecart'])
    print('Synthèse:', (d.get('controles') or {}).get('synthese'))
else:
    print('Aucun JSON.')